# Notebook 12 - Cartas de discusion — Dynamic Augmented Adaptive Group Counting

Cada carta es un ejemplo chico con su figura y un par de preguntas abiertas,
pensado para discutir y abrir direcciones. Poco texto: la figura manda.


In [ ]:
%matplotlib inline
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isfile(os.path.join(_d, "augmented", "__init__.py")):
    _d = os.path.dirname(_d)
if not os.path.isfile(os.path.join(_d, "augmented", "__init__.py")):
    _fb = "/Users/hectorbecerrilvillamil/Desktop/GroupCounting/group-count-dynamic"
    if os.path.isfile(os.path.join(_fb, "augmented", "__init__.py")):
        _d = _fb
if _d not in sys.path:
    sys.path.insert(0, _d)
import numpy as np
import matplotlib.pyplot as plt
import random
random.seed(0); np.random.seed(0)


## Greedy vs óptimo

n=4, B=2, G=3. El greedy gasta su primer test en una sola persona (un pool de 1) y saca 3.700. El óptimo consulta un grupo de dos y, según cuántos activos reporte el conteo (0, 1 o 2), elige distinto el segundo test; llega a 4.187. La brecha es (4.187 − 3.700) / 4.187 = 11.6%.

In [ ]:
import io
from augmented.solver import solve_optimal_dapts
from augmented.greedy import greedy_myopic_counting_expected_utility, _myopic_best_pool
from augmented.bayesian import bayesian_update_single_test
from augmented.tree_extractor import extract_tree
from augmented.tree_visualizer import render_tree

# Instancia n=4, B=2, G=3 donde el greedy miope queda por debajo del optimo augmented.
n, B, G = 4, 2, 3
p = [0.552, 0.122, 0.474, 0.269]
u = [2.735, 2.817, 2.332, 0.82]

# Optimo augmented: DP exacta que ramifica sobre el conteo de activos.
U_opt, pol = solve_optimal_dapts(p, u, B, G)
t_opt = extract_tree(pol, p, u, n)

# El greedy no tiene policy: lo envolvemos con la misma firma choose(k, history).
# Re-jugamos la historia para recuperar cleared_mask y el posterior secuencial,
# y devolvemos el pool miope (el mismo _myopic_best_pool que usa el repo).
class GreedyPolicy:
    def __init__(self, p, u, G, n, B):
        self.p, self.u, self.G, self.n, self.B = p, u, G, n, B
    def choose(self, k, history):
        cur, cleared = list(self.p), 0
        for pool, r in history:
            if pool == 0:
                continue
            if r == 0:
                cleared |= pool
            cur = bayesian_update_single_test(cur, pool, r, self.n)
        return _myopic_best_pool(cur, self.u, self.G, self.n, cleared)

t_gre = extract_tree(GreedyPolicy(p, u, G, n, B), p, u, n)
U_gre = greedy_myopic_counting_expected_utility(p, u, B, G)
gap = (U_opt - U_gre) / U_opt

# render_side_by_side devuelve HTML con SVGs; para una sola figura PNG inline
# rendereamos cada arbol con render_tree y los componemos en un matplotlib figure.
dot_gre = render_tree(t_gre, n, title=f"Greedy miope   U={U_gre:.3f}")
dot_opt = render_tree(t_opt, n, title=f"Optimo augmented   U={U_opt:.3f}")
img_gre = plt.imread(io.BytesIO(dot_gre.pipe(format="png")), format="png")
img_opt = plt.imread(io.BytesIO(dot_opt.pipe(format="png")), format="png")

fig, axes = plt.subplots(1, 2, figsize=(15, 9))
for ax, img in zip(axes, (img_gre, img_opt)):
    ax.imshow(img); ax.axis("off")
fig.suptitle(f"Arboles de decision  (n=4, B=2, G=3)   gap = {gap*100:.1f}%  "
             f"(U_opt={U_opt:.3f} vs U_greedy={U_gre:.3f})", fontsize=13, y=0.99)
fig.tight_layout()
plt.show()

print(f"U_opt={U_opt:.4f}  U_greedy={U_gre:.4f}  gap={gap*100:.2f}%")
print(f"primer pool greedy = {bin(GreedyPolicy(p,u,G,n,B).choose(1,()))}, "
      f"primer pool optimo = {bin(pol.choose(1,()))}")

## Información cruzada

Dos tests comparten a la persona 1: t1={0,1} da 1 activo y t2={1,2} da 0. Como t2 dice que 1 y 2 están limpios, el activo de t1 tiene que ser el 0, así que su probabilidad sube a 1.0. Si actualizas test por test (sin cruzar la información), el 0 no aparece en t2 y se queda en su prior 0.3.

In [ ]:
from matplotlib.patches import Ellipse
from augmented.bayesian import bayesian_update, bayesian_update_by_counting

# Ejemplo 1 (sec 3.1): n=3, dos tests con conteo exacto
n = 3
p = [0.3, 0.5, 0.2]
t1 = (1 << 0) | (1 << 1)   # pool {0,1}
t2 = (1 << 1) | (1 << 2)   # pool {1,2}
history = ((t1, 1), (t2, 0))

seq = bayesian_update(p, history, n)                # (a) secuencial / local
joint = bayesian_update_by_counting(p, history, n)  # (b) conjunto exacto

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.6))

# Panel izquierdo: los dos pools como elipses que se solapan en el individuo 1
axL.set_xlim(0, 10); axL.set_ylim(0, 10); axL.axis("off")
axL.set_title("Dos pools que se solapan en el individuo 1", fontsize=11)
pos = {0: (2.6, 5.0), 1: (5.0, 5.0), 2: (7.4, 5.0)}
axL.add_patch(Ellipse((3.8, 5.0), 4.6, 2.6, facecolor="#cfe3f7",
                      edgecolor="#2c6fbb", lw=2, alpha=0.55, zorder=1))
axL.add_patch(Ellipse((6.2, 5.0), 4.6, 2.6, facecolor="#f7d8cf",
                      edgecolor="#bb472c", lw=2, alpha=0.55, zorder=1))
axL.text(2.3, 7.0, "t1 = {0,1},  r = 1", color="#2c6fbb", fontsize=11, ha="center")
axL.text(7.7, 7.0, "t2 = {1,2},  r = 0", color="#bb472c", fontsize=11, ha="center")
for i, (x, y) in pos.items():
    axL.scatter([x], [y], s=900, color="white", edgecolors="black", lw=1.6, zorder=3)
    axL.text(x, y, str(i), ha="center", va="center", fontsize=13, zorder=4)
    axL.text(x, y - 1.0, f"p={p[i]:.1f}", ha="center", va="center", fontsize=9, color="#444")
axL.text(5.0, 2.6,
         "t2 dice 0 activos en {1,2}  ->  1 y 2 limpios\n"
         "t1 dice 1 activo en {0,1}  ->  el activo es el 0",
         ha="center", va="center", fontsize=9.5, color="#333",
         bbox=dict(boxstyle="round,pad=0.4", fc="#fbfbe8", ec="#cccc99"))

# Panel derecho: barras agrupadas del posterior P(estado activo)
idx = np.arange(n)
w = 0.38
bars_seq = axR.bar(idx - w/2, seq, w, label="Secuencial (local)",
                   color="#8aa9c9", edgecolor="black", lw=0.8)
bars_joint = axR.bar(idx + w/2, joint, w, label="Conjunto (exacto)",
                     color="#c97f6e", edgecolor="black", lw=0.8)
axR.set_xticks(idx); axR.set_xticklabels([f"ind. {i}" for i in idx])
axR.set_ylabel("P(activo | historia)")
axR.set_ylim(0, 1.18)
axR.set_title("Posterior por individuo: secuencial vs conjunto", fontsize=11)
axR.legend(loc="upper right", fontsize=9)
for b in list(bars_seq) + list(bars_joint):
    h = b.get_height()
    axR.text(b.get_x() + b.get_width()/2, h + 0.02, f"{h:.2f}",
             ha="center", va="bottom", fontsize=9)
axR.annotate("", xy=(0 + w/2, joint[0]), xytext=(0 - w/2, seq[0]),
             arrowprops=dict(arrowstyle="->", color="#b5179e", lw=2.2))
axR.text(0.05, 0.66, "cross-information\n0.30 -> 1.00", color="#b5179e",
         fontsize=9.5, ha="left", va="center")

fig.tight_layout()
plt.show()

print("seq  =", [round(x, 4) for x in seq])
print("joint=", [round(x, 4) for x in joint])

## Super-nodo

Testeo el grupo {3,6} y sale 1 activo: eso amarra a 3 y 6, porque ahora exactamente uno de los dos lo está. Sus probabilidades dejan de ser independientes, así que ya no puedes tratarlas por separado y multiplicarlas; las junto en un solo super-nodo S y trabajo con su probabilidad conjunta.

In [ ]:
from matplotlib.patches import Ellipse
from augmented.bayesian import bayesian_update_by_counting

# --- Instancia (misma del ejemplo VW) ---
POS = {
    0: (3.2, 1.0), 1: (3.0, 5.7), 2: (5.4, 8.1), 3: (6.7, 5.2), 4: (3.2, 3.4),
    5: (4.8, 6.6), 6: (5.7, 3.6), 7: (7.4, 7.2), 8: (1.9, 4.2),
    9: (6.6, 1.2), 10: (1.4, 6.2), 11: (1.7, 8.4), 12: (3.3, 8.5),
}
POOLS = {"t1": [2, 5], "t2": [1, 4], "t3": [3, 6], "t4": [0, 4, 6, 9]}
N = len(POS)
POOL_COLORS = {"t1": "#1d6fb8", "t2": "#2a9d8f", "t3": "#e76f51", "t4": "#8e44ad"}

rng = np.random.default_rng(7)
p = {i: round(float(rng.uniform(0.15, 0.55)), 2) for i in POS}   # prior P(estado activo)
u = {i: int(rng.integers(1, 4)) for i in POS}                    # utilidad


def mask(ids):
    m = 0
    for i in ids:
        m |= (1 << i)
    return m


def ellipse_params(ids, pad=1.2):
    pts = np.array([POS[i] for i in ids], float)
    c = pts.mean(0)
    if len(pts) == 1:
        return c, 1.0, 1.0, 0.0
    d = pts - c
    vals, vecs = np.linalg.eigh(np.cov(d.T) + 1e-6 * np.eye(2))
    proj = d @ vecs
    h = 2 * np.abs(proj[:, 0]).max() + pad
    w = 2 * np.abs(proj[:, 1]).max() + pad
    ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    return c, w, h, ang


def draw_pool(ax, ids, color, lw=2.2, alpha=0.12):
    c, w, h, ang = ellipse_params(ids)
    ax.add_patch(Ellipse(c, w, h, angle=ang, facecolor=color, alpha=alpha,
                         edgecolor=color, lw=lw, zorder=1))
    return c


def draw_supernode(ax, ids):
    c, w, h, ang = ellipse_params(ids, pad=1.9)
    ax.add_patch(Ellipse(c, w, h, angle=ang, facecolor="none",
                         edgecolor="k", lw=2.4, ls=(0, (4, 3)), zorder=2))
    ax.text(c[0], c[1] - h / 2 - 0.25, "super-nodo S = {3, 6}", ha="center",
            fontsize=11, fontstyle="italic", zorder=5)


def draw_agents(ax, post):
    for i, (x, y) in POS.items():
        col = plt.cm.RdYlGn_r(post[i])
        ax.scatter([x], [y], s=270, c=[col], edgecolors="k", linewidths=1.2, zorder=3)
        ax.text(x, y, str(i), ha="center", va="center", fontsize=9, zorder=4)


# --- Posterior tras la historia ((mask{3,6}, 1),) ---
S = [3, 6]
history = ((mask(S), 1),)
q_vec = bayesian_update_by_counting([p[i] for i in range(N)], history, N)
q = {i: q_vec[i] for i in range(N)}

fig, ax = plt.subplots(figsize=(8, 6.5))
for name, ids in POOLS.items():
    c = draw_pool(ax, ids, POOL_COLORS[name])
    lbl = name + "  <- testeado" if name == "t3" else name
    ax.text(c[0], c[1] + 0.1, lbl, color=POOL_COLORS[name],
            fontsize=12, fontweight="bold", ha="center", zorder=5)
draw_supernode(ax, S)
draw_agents(ax, q)

sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn_r, norm=plt.Normalize(0, 1))
fig.colorbar(sm, ax=ax, fraction=0.04, pad=0.02, label="posterior P(estado activo)")
ax.set_title("Super-nodo S tras testear t3 = {3, 6} con conteo r = 1")
ax.set_xlim(0.3, 8.4); ax.set_ylim(0, 9.3); ax.set_aspect("equal"); ax.axis("off")
fig.tight_layout()
plt.show()

print("prior p sobre S:", {i: p[i] for i in S})
print("posterior q sobre S:", {i: round(q[i], 3) for i in S})
print("suma de posteriores en S (= r):", round(q[3] + q[6], 3))

## La utilidad solo se obtiene con el pool completamente conteo-cero

En este esquema solo ganas cuando un pool sale con conteo 0 (todos limpios): ahi declaras limpios a todos sus miembros y te llevas su utilidad. La probabilidad de ese evento todo-limpio es el producto de los q (q_i = 1 - p_i = probabilidad de estar limpio), asi que el valor de testear un pool = P(todo limpio) por la suma de utilidades. Como es un producto, una sola persona riesgosa lo hunde: con p = [0.1, 0.2, 0.3, 0.5, 0.7] y u = [2, 2, 3, 3, 4], el pool {A,B,C,D} tiene P(todo limpio) = 0.252 y valor 2.52, pero meter a E (q = 0.3) lo desploma a P = 0.076 y valor 1.06; el mejor pool encontrado, {A,B,C}, llega a P = 0.504 y valor 3.53.

In [ ]:
from matplotlib.patches import Ellipse

# --- Instancia concreta ---
p = np.array([0.1, 0.2, 0.3, 0.5, 0.7])   # prob de estado latente
q = 1.0 - p                                 # prob de estar SANO = [0.9,0.8,0.7,0.5,0.3]
u = np.array([2, 2, 3, 3, 4])              # utilidad por persona
names = ["A", "B", "C", "D", "E"]          # E es la riesgosa (q=0.3)

def val(idx):
    P = float(np.prod(q[idx]))             # P(todo limpio) = producto de los q
    return P, P * float(np.sum(u[idx]))    # valor = P(todo limpio) * suma de utilidades

pos = {0:(1.3,4.2), 1:(2.7,4.6), 2:(2.0,3.3), 3:(3.6,3.5), 4:(5.0,4.0)}
cmap = plt.cm.RdYlGn                        # q alto -> verde, q bajo -> rojo

fig, ax = plt.subplots(figsize=(11,7))
ax.set_xlim(0,11); ax.set_ylim(0,7.2); ax.axis("off")

ax.text(5.5,6.9,"Solo ganas la utilidad cuando el pool sale COMPLETAMENTE NEGATIVO (0 activos)",
        ha="center",fontsize=13,weight="bold")
ax.text(5.5,6.45,r"P(todo limpio) = $\prod q_i$       valor del pool = P(todo limpio) $\times$ $\sum u_i$",
        ha="center",fontsize=12)

# Pool SIN la riesgosa E: A,B,C,D
P1,V1 = val([0,1,2,3])
ax.add_patch(Ellipse((2.4,3.9),3.6,2.4,fc="none",ec="#1f6f1f",lw=2.8))
ax.text(1.7,5.25,"Pool SIN la riesgosa\n(A,B,C,D)",ha="center",fontsize=11,color="#1f6f1f",weight="bold")
ax.text(2.4,1.95,"P(todo limpio)=%.3f\nvalor=%.2f"%(P1,V1),ha="center",fontsize=11,
        color="#1f6f1f",bbox=dict(boxstyle="round",fc="#eaffea",ec="#1f6f1f"))

# Pool CON la riesgosa E: A,B,C,D,E
P2,V2 = val([0,1,2,3,4])
ax.add_patch(Ellipse((3.1,4.0),5.4,3.2,fc="none",ec="#b22222",lw=2.8,ls="--"))
ax.text(5.2,6.0,"Pool que INCLUYE a la riesgosa E",ha="center",fontsize=11,color="#b22222",weight="bold")
ax.text(6.7,2.4,"P(todo limpio)=%.3f\nvalor=%.2f"%(P2,V2),ha="center",fontsize=11,
        color="#b22222",bbox=dict(boxstyle="round",fc="#ffecec",ec="#b22222"))

# personas como puntos coloreados por su q
for i in range(5):
    x,y = pos[i]
    ax.scatter([x],[y],s=900,c=[cmap(q[i])],edgecolors="black",zorder=5)
    ax.text(x,y,names[i],ha="center",va="center",fontsize=13,weight="bold",zorder=6)
    ax.text(x,y-0.42,"q=%.1f"%q[i],ha="center",va="center",fontsize=9,zorder=6)
    ax.text(x,y+0.42,"u=%d"%u[i],ha="center",va="center",fontsize=9,color="#444",zorder=6)

ax.annotate("",xy=pos[4],xytext=(7.4,4.7),
            arrowprops=dict(arrowstyle="->",color="#b22222",lw=2.2))
ax.text(7.5,4.95,"E es riesgosa (q=0.3):\nhunde el producto",ha="left",fontsize=10,color="#b22222")

# mejor jugada encontrada: pool chico {A,B,C}
Pc,Vc = val([0,1,2])
ax.text(9.1,3.6,"Mejor jugada\nencontrada:",ha="center",fontsize=11,weight="bold")
ax.text(9.1,2.7,"Pool {A,B,C}\nP(todo limpio)=%.3f\nvalor=%.2f"%(Pc,Vc),ha="center",fontsize=11,
        bbox=dict(boxstyle="round,pad=0.5",fc="#fff7cc",ec="#caa400",lw=2.5))
ax.text(9.1,1.4,"Meter a E baja el valor\nde %.2f a %.2f"%(V1,V2),ha="center",fontsize=10,color="#b22222")
ax.text(0.4,0.5,"verde = limpio-probable (q alto)   rojo = riesgoso (q bajo)",
        ha="left",fontsize=10,color="#333")

plt.tight_layout()
plt.show()

## Los perfiles válidos se parten en dos

Cada test devuelve el conteo exacto del pool. Los perfiles válidos son todos los perfiles de estado latente compatibles con esas cuentas. Con dos pools solapados, personas {0,1,2} con conteo 1 y personas {2,3,4} con conteo 1 sobre 5 personas, solo hay 5 perfiles validos: {2} con 1 activo en total, y {0,3}, {0,4}, {1,3}, {1,4} con 2 activos en total. Esos perfiles se parten en dos grupos por el numero total de activos. El movimiento de Gibbs cambia a una persona a la vez intercambiando un limpio por un activo para no romper las cuentas, asi que el total nunca cambia: se atora dentro de un grupo y nunca cruza al otro.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

GREEN="#2e9e54"; RED="#d23b3b"
fig, ax = plt.subplots(figsize=(11.5, 6.8))
ax.set_xlim(0, 11.5); ax.set_ylim(-0.6, 6.4); ax.axis("off")

def perfil(ax, cx, cy, active, label, dot_r=0.13, gap=0.42):
    xs = [cx + (i - 2)*gap for i in range(5)]
    for i, x in enumerate(xs):
        col = RED if i in active else GREEN
        ax.add_patch(plt.Circle((x, cy), dot_r, color=col, zorder=4, ec="white", lw=0.8))
    ax.text(cx, cy - 0.34, label, ha="center", va="top", fontsize=9.5, color="#333")
    return xs

# Caja izquierda: perfiles con 1 activo en total
ax.add_patch(FancyBboxPatch((0.35, 0.7), 2.85, 4.7, boxstyle="round,pad=0.02,rounding_size=0.18",
                            fc="#eef6ff", ec="#5b87b5", lw=2.0, zorder=1))
ax.text(1.78, 5.65, "1 activo en total", ha="center", va="center",
        fontsize=12.5, fontweight="bold", color="#1f4e79")
perfil(ax, 1.78, 3.0, [2], "activo: persona 2")

# Caja derecha: perfiles con 2 activos en total
ax.add_patch(FancyBboxPatch((4.55, 0.4), 6.55, 5.3, boxstyle="round,pad=0.02,rounding_size=0.18",
                            fc="#fff2ef", ec="#c66", lw=2.0, zorder=1))
ax.text(7.83, 5.95, "2 activos en total", ha="center", va="center",
        fontsize=12.5, fontweight="bold", color="#8a2f2f")
perfil(ax, 6.3, 4.35, [0,3], "activos: 0 y 3")
perfil(ax, 9.4, 4.35, [0,4], "activos: 0 y 4")
perfil(ax, 6.3, 2.15, [1,3], "activos: 1 y 3")
perfil(ax, 9.4, 2.15, [1,4], "activos: 1 y 4")

# Flechas swap: cada par difiere en un solo intercambio limpio<->activo
def swap_arrow(ax, a, b, text):
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle="<|-|>", mutation_scale=12,
                                 color="#7a4a4a", lw=1.4, zorder=3))
    mx, my = (a[0]+b[0])/2, (a[1]+b[1])/2
    ax.text(mx, my, text, ha="center", va="center", fontsize=8.5, color="#7a4a4a",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))

swap_arrow(ax, (7.25, 4.35), (8.45, 4.35), "swap 3 a 4")
swap_arrow(ax, (7.25, 2.15), (8.45, 2.15), "swap 3 a 4")
swap_arrow(ax, (6.3, 3.7), (6.3, 2.8), "swap 0 a 1")
swap_arrow(ax, (9.4, 3.7), (9.4, 2.8), "swap 0 a 1")

# Barrera gruesa con X entre las dos cajas
bx = 3.85
ax.plot([bx, bx], [0.5, 5.6], color="#444", lw=5, zorder=5)
for s in (-1, 1):
    ax.plot([bx-0.28, bx+0.28], [3.05 + s*0.28, 3.05 - s*0.28], color=RED, lw=4, zorder=6,
            solid_capstyle="round")
ax.annotate("Gibbs intercambia un limpio por un activo:\nel total nunca cambia, no cruza la barrera",
            xy=(bx, 0.55), xytext=(bx, -0.35), ha="center", va="center", fontsize=9.5, color="#333",
            bbox=dict(boxstyle="round,pad=0.3", fc="#fffbe6", ec="#b59f3b", lw=1.3),
            arrowprops=dict(arrowstyle="-", color="#b59f3b", lw=1.0))

# Leyenda de puntos
ax.add_patch(plt.Circle((0.7, 6.05), 0.11, color=GREEN, ec="white", lw=0.8))
ax.text(0.92, 6.05, "limpio", va="center", fontsize=9.5)
ax.add_patch(plt.Circle((2.0, 6.05), 0.11, color=RED, ec="white", lw=0.8))
ax.text(2.22, 6.05, "activo", va="center", fontsize=9.5)

ax.set_title("Los perfiles validos se parten en dos por el total de activos",
             fontsize=13, pad=8)
plt.tight_layout()
plt.show()

## Tratabilidad: el costo de calcular el posterior exacto depende de como se cruzan los pools

Cada test devuelve el conteo de activos de su pool. Para decidir a quien declarar limpio calculamos el posterior exacto: la probabilidad de cada combinacion posible de limpios e activos. Ese calculo es barato cuando los pools casi no comparten personas, porque se puede resolver por partes (un grupo separado, o de adentro hacia afuera, o de eslabon en eslabon). Se vuelve caro solo cuando todos los pools se solapan entre si: ahi no hay atajo y hay que recorrer todas las combinaciones. Con 6 personas muy cruzadas ya son 2^6 = 64 estados; con 30 serian mas de mil millones, y crece al doble por cada persona extra.

In [ ]:
from matplotlib.patches import Ellipse

fig, axes = plt.subplots(1, 4, figsize=(15.5, 4.6))

VERDE = "#1b7a3d"
ROJO  = "#c0392b"
AZUL  = "#2c3e50"
PUNTO = "#34495e"

def persona(ax, x, y):
    ax.plot(x, y, "o", color=PUNTO, ms=11, zorder=5)

def elipse(ax, cx, cy, w, h, ang=0, color="#1f6feb"):
    e = Ellipse((cx, cy), w, h, angle=ang, fill=False,
                edgecolor=color, lw=2.6, zorder=3)
    ax.add_patch(e)

def marco(ax, titulo, etiqueta, rapido, motivo):
    col = VERDE if rapido else ROJO
    ax.set_title(titulo, fontsize=13, fontweight="bold", color=AZUL, pad=10)
    ax.text(0.5, 0.92, etiqueta, transform=ax.transAxes, ha="center", va="top",
            fontsize=14, fontweight="bold", color=col,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=col, lw=2))
    ax.text(0.5, -0.16, motivo, transform=ax.transAxes, ha="center", va="top",
            fontsize=10.5, color=col)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor(col); s.set_linewidth(2.2)

# --- Caso 1: pools SEPARADOS ---
ax = axes[0]
for x in (2.2, 3.4): persona(ax, x, 6.5)
for x in (6.8, 8.0): persona(ax, x, 6.5)
for x in (2.2, 3.4): persona(ax, x, 3.2)
elipse(ax, 2.8, 6.5, 2.6, 2.0)
elipse(ax, 7.4, 6.5, 2.6, 2.0)
elipse(ax, 2.8, 3.2, 2.6, 2.0)
ax.text(7.4, 3.2, "cada grupo\nsuelto", ha="center", va="center",
        fontsize=10, color=AZUL, style="italic")
marco(ax, "Pools SEPARADOS", "RAPIDO", True,
      "No comparten personas:\ncada grupo se resuelve por su cuenta.")

# --- Caso 2: pools ANIDADOS ---
ax = axes[1]
for x in (4.3, 5.0, 5.7): persona(ax, x, 5.0)
persona(ax, 5.0, 6.6)
persona(ax, 5.0, 3.4)
elipse(ax, 5.0, 5.0, 2.2, 1.6)
elipse(ax, 5.0, 5.0, 5.2, 5.6)
marco(ax, "Pools ANIDADOS", "RAPIDO", True,
      "Uno dentro de otro:\nse resuelve de adentro hacia afuera.")

# --- Caso 3: pools EN CADENA ---
ax = axes[2]
xs = [2.0, 3.5, 5.0, 6.5, 8.0]
for x in xs: persona(ax, x, 5.0)
elipse(ax, 2.75, 5.0, 2.6, 2.0)
elipse(ax, 4.25, 5.0, 2.6, 2.0)
elipse(ax, 5.75, 5.0, 2.6, 2.0)
elipse(ax, 7.25, 5.0, 2.6, 2.0)
marco(ax, "Pools EN CADENA", "RAPIDO", True,
      "Cada pool comparte solo con el de al lado:\nse cruzan poco.")

# --- Caso 4: pools MUY CRUZADOS ---
ax = axes[3]
ang = np.linspace(0, 2*np.pi, 6, endpoint=False) + np.pi/2
px, py = 5 + 1.9*np.cos(ang), 5 + 1.9*np.sin(ang)
for x, y in zip(px, py): persona(ax, x, y)
for a, c in zip((0, 60, 120), ("#1f6feb", "#9b59b6", "#e67e22")):
    elipse(ax, 5, 5, 7.0, 3.0, ang=a, color=c)
marco(ax, "Pools MUY CRUZADOS", "LENTO", False,
      "Todos se solapan:\nhay que mirar TODAS las combinaciones.")

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()